# Project 2: Complete ML Pipeline

**Goal:** End-to-end machine learning workflow
- Load data → Clean → Split → Scale → Train multiple models → Evaluate → Select best

**Models Compared:**
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. SVM
5. KNN

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
# Load data (using iris for simplicity, but you can use any dataset)
from sklearn.datasets import load_iris
iris = load_iris()
X, y = iris.data, iris.target

# Or create synthetic data
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)

# Convert to DataFrame for exploration
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y

print("Shape:", df.shape)
print("\nFirst 5 rows:\n", df.head())
print("\nTarget distribution:\n", df['target'].value_counts())

Shape: (150, 5)

First 5 rows:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

Target distribution:
 target
0    50
1    50
2    50
Name: count, dtype: int64


In [3]:
# Step 1: Split data
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (120, 4), Test: (30, 4)


In [4]:
# Step 2: Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# Step 3: Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier()
}

# Step 4: Train and evaluate with cross-validation
results = {}

for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    # Train on full training set
    model.fit(X_train_scaled, y_train)
    
    # Test set prediction
    test_pred = model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test, test_pred)
    
    results[name] = {
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': test_acc
    }
    
    print(f"{name}:")
    print(f"  CV: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    print(f"  Test: {test_acc:.4f}")
    print()

Logistic Regression:
  CV: 0.9583 (+/- 0.0527)
  Test: 0.9333

Decision Tree:
  CV: 0.9417 (+/- 0.0408)
  Test: 0.9333

Random Forest:
  CV: 0.9500 (+/- 0.0333)
  Test: 0.9000

SVM:
  CV: 0.9667 (+/- 0.0624)
  Test: 0.9667

KNN:
  CV: 0.9667 (+/- 0.0624)
  Test: 0.9333

